# Experiments

## Setup: Import Libraries and Scripts

In [ ]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

SyntaxError: invalid syntax (130714823.py, line 3)

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
    {
        'name': 'Zero-Shot Baseline (Default)',
        'script': 'zero_shot',
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'test_sample_size': 100,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Zero-Shot No Contexts',
        'script': 'zero_shot',
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'test_sample_size': 50,
        'statutory_context_enabled': False,
        'contract_context_enabled': False
    },
    {
        'name': 'Full Optimization (Small)',
        'script': 'optimize',
        'generations': 5,
        'pop_size': 4,
        'train_sample_size': 20,
        'test_sample_size': 50,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization (No Bandits, No Statutory)',
        'script': 'optimize',
        'generations': 3,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 50,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': False,
        'contract_context_enabled': True
    }
]

## Run Experiments

This cell runs each experiment and collects results.

In [ ]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report']  # Raw dict if needed
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e)})

# Create DataFrame
df_results = pd.DataFrame(results)

## Display Results Table

Interactive table with all parameters and metrics. Scroll horizontally if needed.

In [ ]:
if not df_results.empty:
    # Style the table for better readability
    styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
        {'selector': 'th', 'props': [('text-align', 'left')]}
    ]).background_gradient(cmap='viridis', subset=['F1 Macro'])
    display(HTML("<h3>Experiment Results</h3>"))
    display(styled_df)
else:
    print("No results to display.")